# Lab: Train an ML model on ERA5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ml_play/train_era5.ipynb)

Can a small neural network learn tomorrow's weather from today's? Here you train one on 10 years of daily ERA5 500 hPa geopotential height (Z500): it sees one day's map and predicts the next day's.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU): training takes about a minute there and much longer on a CPU. Run the two setup cells first.

In [ ]:
#@title Setup: install packages (~1 min)
!pip install -q gcsfs zarr cartopy

In [ ]:
#@title Setup: imports and ERA5 data
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr
from torch import nn

# Daily means of ERA5 from 1959 to 2022 on a 1.5 degree grid (240 x 121 points).
# This is WeatherBench 2's copy on Google Cloud; no account needed.
era5 = xr.open_zarr("gs://weatherbench2/datasets/era5_daily/1959-2023_01_10-1h-240x121_equiangular_with_poles_conservative.zarr",
                    storage_options=dict(token="anon"))
era5 = era5.transpose(..., "latitude", "longitude").sortby("latitude")   # stored as (time, lon, lat); put latitude first, south to north

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
print("Training on", device)

---
## 1. The data

Z500 is the height of the 500 hPa pressure surface, about halfway up through the atmosphere's mass. Its highs (ridges) and lows (troughs) are the weather systems that move around the globe from day to day.

In [ ]:
# 10 years of daily Z500 (~3 GB to download, a few minutes). Geopotential / g = height in meters.
z500 = era5["geopotential"].sel(level=500, time=slice("2010", "2019")).compute() / 9.80665
print(f"{z500.sizes['time']} days, {z500.sizes['latitude']} x {z500.sizes['longitude']} grid points")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), layout="constrained", subplot_kw=dict(projection=ccrs.Robinson(central_longitude=180)))

for ax, day in zip(axes, ["2015-01-15", "2015-01-16"]):
    mesh = ax.pcolormesh(z500.longitude, z500.latitude, z500.sel(time=day), transform=ccrs.PlateCarree(), vmin=4900, vmax=5900)
    ax.coastlines(linewidth=0.5)
    ax.set_title(f"Z500 on {day}")
fig.colorbar(mesh, ax=axes, shrink=0.8, label="m")
plt.show()

The model's input is one day's map and its target is the next day's. Neural networks train best on numbers of order one, so the heights are standardized first (minus their mean, divided by their standard deviation).

In [ ]:

mean, std = float(z500.mean()), float(z500.std())

z = torch.tensor(((z500 - mean) / std).values, dtype=torch.float32)[:, None]   # (days, 1 channel, latitude, longitude)

x, y = z[:-1], z[1:]   # input: today; target: tomorrow

print("input:", tuple(x.shape), " target:", tuple(y.shape))

---
## 2. The model

A small convolutional neural network (CNN): three layers of 5 x 5 filters, each looking at a point and its neighbors. Stacked, they see about 20° around each point, roughly how far weather systems move in a day. "Circular" padding wraps the map around in longitude so there is no edge at the date line.

Tomorrow looks a lot like today, so the network predicts only the change, and the forecast is today plus that change.

In [ ]:
# create a simple sequential model with three convolutional layers
model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1, padding_mode="circular"), nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1, padding_mode="circular"), nn.ReLU(),
    nn.Conv2d(32, 1, kernel_size=3, padding=1, padding_mode="circular"),
).to(device)

# print the parameter count
print(f"{sum(p.numel() for p in model.parameters())} parameters")

EPOCHS = 10       # passes through all the days
BATCH_SIZE = 32   # days per training step
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(EPOCHS):
    order = torch.randperm(len(x))   # a new random order of the days each pass
    for i in range(0, len(x), BATCH_SIZE):
        batch = order[i:i + BATCH_SIZE]
        today, tomorrow = x[batch].to(device), y[batch].to(device)
        forecast = today + model(today)
        loss = ((forecast - tomorrow) ** 2).mean()   # mean squared error
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    print(f"epoch {epoch + 1}: loss {np.mean(losses[-len(x) // BATCH_SIZE:]):.4f}")

persistence = ((x - y) ** 2).mean().item()   # the loss of just saying "tomorrow = today"
plt.figure(figsize=(8, 3))
plt.plot(losses, color="black", linewidth=0.5, label="CNN")
plt.axhline(persistence, color="tab:red", label="tomorrow = today")
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("mean squared error")
plt.legend()
plt.show()

---
## 3. How good is it?

Compare the CNN with the simplest forecast of all, persistence: tomorrow will be the same as today. Errors are root mean square, in meters, weighted by area.

In [ ]:
with torch.no_grad():
    forecast = torch.cat([x[i:i + 256].to(device) + model(x[i:i + 256].to(device)) for i in range(0, len(x), 256)])
forecast = forecast[:, 0].cpu().numpy() * std + mean   # back to meters
today, tomorrow = z500.values[:-1], z500.values[1:]

area = np.cos(np.deg2rad(z500.latitude.values))[:, None] * np.ones(z500.sizes["longitude"])
rmse_cnn = np.sqrt(np.average(((forecast - tomorrow) ** 2).mean(axis=0), weights=area))
rmse_persistence = np.sqrt(np.average(((today - tomorrow) ** 2).mean(axis=0), weights=area))
print(f"RMSE of tomorrow = today: {rmse_persistence:.1f} m")
print(f"RMSE of the CNN:          {rmse_cnn:.1f} m")

Look at one day. The bottom row is the change from today to tomorrow: what actually happened, and what the CNN predicted.

In [ ]:
DAY = "2015-01-15"   # any day from 2010-01-01 to 2019-12-30

i = z500.indexes["time"].get_loc(DAY)
maps = [(today[i], f"Today, {DAY}"), (tomorrow[i], "Tomorrow: ERA5"), (forecast[i], "Tomorrow: CNN"),
        (tomorrow[i] - today[i], "Change: ERA5"), (forecast[i] - today[i], "Change: CNN")]
top = np.percentile(np.abs(tomorrow[i] - today[i]), 99)

fig = plt.figure(figsize=(16, 6.5), layout="constrained")
axes = []
for j, (values, title) in enumerate(maps):
    ax = fig.add_subplot(2, 3, j + 1 + (j >= 3), projection=ccrs.Robinson(central_longitude=180))
    if j < 3:
        mesh1 = ax.pcolormesh(z500.longitude, z500.latitude, values, transform=ccrs.PlateCarree(), vmin=4900, vmax=5900)
    else:
        mesh2 = ax.pcolormesh(z500.longitude, z500.latitude, values, transform=ccrs.PlateCarree(), cmap="RdBu_r", vmin=-top, vmax=top)
    ax.coastlines(linewidth=0.5)
    ax.set_title(title)
    axes.append(ax)
fig.colorbar(mesh1, ax=axes[:3], shrink=0.8, label="Z500 (m)")
fig.colorbar(mesh2, ax=axes[3:], shrink=0.8, label="change (m)")
plt.show()

Things to try:
- Train for more epochs, or make the network wider (more than 32 filters) or deeper. Does the error keep going down?
- The model has only ever been graded on the days it trained on. Train on 2010–2018 and test on 2019: is it still better than persistence?
- Feed the forecast back in as the next input to forecast 2, 3, 10 days ahead. What happens to the maps?